In [18]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
from sklearn import tree
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 


# Combine X and y horizontally
adult_df = pd.concat([X, y], axis=1)

# Preview the result
adult_df.head() 


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [2]:
# 1. What is inductive reasoning? Deductive reasoning? Give an example of each, different from the examples given in class
#Inductive reasoning is when we go from a specific observation to general conclusions -- example: I see 4 cats that have a tail, so I conclude that all cats have tails 
#deductive reasoning is when we go from a general theory to a specific conclusion -- example: All cats have tails, my pet has a tail, therefore my pet is a cat

In [3]:
# 2. preprocessing
# drop occupation, education, workclass
adult_df = adult_df.drop(columns=['occupation', 'education', 'workclass'])

In [4]:
# convert native country to a true false United States or not
adult_df['native-country_United-States'] = adult_df['native-country'].apply(lambda x: 1 if x == 'United-States' else 0)
adult_df = adult_df.drop(columns=['native-country'])
adult_df.head()

,age,fnlwgt,education-num,marital-status,relationship,race,sex,capital-gain,capital-loss,hours-per-week,income,native-country_United-States
0,39,77516,13,Never-married,Not-in-family,White,Male,2174,0,40,<=50K,1
1,50,83311,13,Married-civ-spouse,Husband,White,Male,0,0,13,<=50K,1
2,38,215646,9,Divorced,Not-in-family,White,Male,0,0,40,<=50K,1
3,53,234721,7,Married-civ-spouse,Husband,Black,Male,0,0,40,<=50K,1
4,28,338409,13,Married-civ-spouse,Wife,Black,Female,0,0,40,<=50K,0


In [5]:
# strip whitspace
adult_df = adult_df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [6]:
# convert sex, race, relationship, marital-status and income to integers
adult_df['sex'] = adult_df['sex'].apply(lambda x: 1 if x == 'Male' else 0)
adult_df['race'] = adult_df['race'].apply(lambda x: 2 if x =='Black' else 1 if x == 'White' else 0)
adult_df['relationship'] = adult_df['relationship'].apply(lambda x: 2 if x == 'Wife' else 1 if x == 'Husband' else 0)
adult_df['marital-status'] = adult_df['marital-status'].apply(lambda x: 2 if x == 'Married-civ-spouse' else 1 if x == 'Never-married' else 0)
adult_df['income'] = adult_df['income'].apply(lambda x: 1 if x == '>50K' else 0)
adult_df.head()

,age,fnlwgt,education-num,marital-status,relationship,race,sex,capital-gain,capital-loss,hours-per-week,income,native-country_United-States
0,39,77516,13,1,0,1,1,2174,0,40,0,1
1,50,83311,13,2,1,1,1,0,0,13,0,1
2,38,215646,9,0,0,1,1,0,0,40,0,1
3,53,234721,7,2,1,2,1,0,0,40,0,1
4,28,338409,13,2,2,2,0,0,0,40,0,0


In [7]:
# 3. Create a decision tree model tuned to the best of your abilities. Explain how you tuned it.
#function to determine optimal depth for decision tree
def optimal_depth(X_train, y_train, X_test, y_test):
    depth_values = range(1, 20)
    best_depth = 1
    best_score = 0
    
    for d in depth_values:
        dt = tree.DecisionTreeClassifier(max_depth=d, random_state=50)
        dt.fit(X_train, y_train)
        score = dt.score(X_test, y_test)
        
        if score > best_score:
            best_score = score
            best_depth = d
            
    return best_depth, best_score

In [8]:
X = adult_df.drop('income',axis=1)
y = adult_df['income']

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.2,
                                                    random_state=50)
optimal_depth, optimal_score = optimal_depth(X_train, y_train, X_test, y_test)
model = tree.DecisionTreeClassifier(max_depth=optimal_depth, random_state=50)
model = model.fit(X_train, y_train)
y_pred = model.predict(X_test)
model.score(X_test, y_test)
# also tinkered with the test size to maximize the accuaracy score

0.8594533729143208

In [ ]:
#4.Create a random forest model tuned to the best of your abilities. Explain how you tuned it.
# function to determine optimal amount of estimators to best tune the random forest 
def optimal_estimators(X_train, y_train, X_test, y_test):
    estimator_values = range(10, 301, 10)
    best_estimator = 1
    best_score = 0
    
    for d in estimator_values:
        dt = RandomForestClassifier(n_estimators=d, random_state=50, n_jobs=-1)
        dt.fit(X_train, y_train)
        score = dt.score(X_test, y_test)
        
        if score > best_score:
            best_score = score
            best_estimator = d
            
    return best_estimator, best_score


In [16]:
best_n, best_s = optimal_estimators(X_train, y_train, X_test, y_test)

In [ ]:
rf = RandomForestClassifier(n_estimators=best_n, random_state=50)
rf.fit(X_train, y_train)
print(rf.score(X_test, y_test))
 

0.8425632101545706


In [ ]:
# 5. Create an xgboost model tuned to the best of your abilities. Explain how you tuned it.
xgb = XGBClassifier()
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
print(xgb.score(X_test, y_test))

0.8582249974408844
